In [ ]:
import json
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp
import analysis as an
from halo_pipeline import require_peculiar_velocities  # halo catalogs made before the velocity fix are refused
sys.path.insert(0, str(hp.HODDIES_ROOT))
for module_name in list(sys.modules):
    if module_name == 'HODDIES' or module_name.startswith('HODDIES.'):
        del sys.modules[module_name]

import numpy as np
import matplotlib.pyplot as plt
from HODDIES import HOD
from HODDIES.sim_loader import Base_catalogue
from abacus_reference_cosmology import SERIALIZABLE_PARAMETERS
import HODDIES

print(f'Using HODDIES from {HODDIES.__file__}')


# Load LRG power spectra (paired-fixed)

In [ ]:
from cosmoprimo.fiducial import Cosmology

abacus_cosmo = Cosmology(
    Omega0_m=SERIALIZABLE_PARAMETERS['Omega_m'],
    Omega_b=SERIALIZABLE_PARAMETERS['Omega_b'],
    h=SERIALIZABLE_PARAMETERS['h'],
    sigma8=SERIALIZABLE_PARAMETERS['sigma8'],
    n_s=SERIALIZABLE_PARAMETERS['n_s'],
    w0_fld=SERIALIZABLE_PARAMETERS['w0'],
    wa_fld=SERIALIZABLE_PARAMETERS['wa'],
    T_cmb=SERIALIZABLE_PARAMETERS['Tcmb0_K'],
    N_eff=SERIALIZABLE_PARAMETERS['Neff'],
    m_ncdm=[],
    engine='class',
)


In [ ]:
from nbodykit.lab import ArrayCatalog, FFTPower


def compute_lrg_pk(cat, boxsize, tracer='LRG', nmesh=256, dk=None, kmin=0.0, subtract_shotnoise=True, centrals_only=False):
    mask = cat['TRACER'] == tracer
    if centrals_only:
        mask = mask & (cat['Central'] == 1)
    pos = np.column_stack([
        cat['x'][mask] % boxsize,
        cat['y'][mask] % boxsize,
        cat['z'][mask] % boxsize,
    ]).astype('f4')

    galaxy_cat = ArrayCatalog({'Position': pos})
    mesh = galaxy_cat.to_mesh(Nmesh=nmesh, BoxSize=boxsize, resampler='tsc',
                               interlaced=True, compensated=True)
    result = FFTPower(mesh, mode='1d', dk=dk or 2 * np.pi / boxsize, kmin=kmin)

    k = result.power['k']
    Pk = result.power['power'].real
    if subtract_shotnoise:
        Pk = Pk - result.power.attrs['shotnoise']

    valid = k > 0  # drop the k=0 DC bin
    return k[valid], Pk[valid]


def load_lrg_pk_for_model(model, realization, boxsize=500.0, redshift=0.500, fix_seed=42, centrals_only=False):
    # Same halo-catalog layout and cosmology-metadata validation as HOD_Abacus.ipynb
    # (cells 2/3/5), parameterised over `model`/`realization` so it also works for the
    # paired-fixed boxes (s1234-fixed / s1234-fixed-inv) -- the reference cosmology (SERIALIZABLE_PARAMETERS) is shared
    # across all Abacus-flavoured runs, not keyed per model.
    param_file = str(hp.NOTEBOOK_DIR / f'HOD_parameter_file_z{redshift:.3f}.yaml')
    output_dir = hp.halo_catalog_dir('Abacus_Small', model, realization, redshift)
    hoddies_catalog = output_dir / 'halos200c_hoddies.npy'
    halo_metadata_file = Path(str(output_dir / 'halos200c_catalog.npy') + '.json')

    if not hoddies_catalog.exists():
        raise FileNotFoundError(
            f'Missing {hoddies_catalog}. Run jobs/halo_finder.sh on '
            f'Abacus_Small/{model}_{realization}.lua first.'
        )

    arr = np.load(hoddies_catalog)

    if halo_metadata_file.exists():
        with halo_metadata_file.open() as f:
            halo_metadata = json.load(f)
        require_peculiar_velocities(halo_metadata, halo_metadata_file)
        halo_cosmology = halo_metadata.get('cosmology')
        if halo_cosmology is not None:
            mismatches = {
                key: (halo_cosmology.get(key), expected)
                for key, expected in SERIALIZABLE_PARAMETERS.items()
                if key in halo_cosmology and not np.isclose(float(halo_cosmology[key]), float(expected))
            }
            if mismatches:
                raise ValueError(f'{model}/{realization}: halo catalog cosmology does not match AbacusSummit c000 reference: {mismatches}')

    base_catalog = Base_catalogue({name: arr[name] for name in arr.dtype.names}, boxsize=boxsize, z_simu=redshift)
    HOD_obj = HOD(base_catalog, param_file=param_file)
    HOD_obj.cosmo = abacus_cosmo  # used for the RSD of get_wp / get_xiells

    cat = HOD_obj.make_mock_cat(tracers='LRG', fix_seed=fix_seed)
    k, Pk = compute_lrg_pk(cat, HOD_obj.boxsize, tracer='LRG', centrals_only=centrals_only)
    return k, Pk


In [ ]:
k_LCDM_a, Pk_LCDM_a = load_lrg_pk_for_model('LCDM', 's1234-fixed')
k_LCDM_ma, Pk_LCDM_ma = load_lrg_pk_for_model('LCDM', 's1234-fixed-inv')
assert np.allclose(k_LCDM_a, k_LCDM_ma), "LCDM_Abacus_a/_ma LRG k-grids differ"
k_LCDM_LRG = k_LCDM_a
Pk_LCDM_LRG_PairedFixed = (Pk_LCDM_a + Pk_LCDM_ma) / 2

k_nDGP_a, Pk_nDGP_a = load_lrg_pk_for_model('nDGP_N1.2', 's1234-fixed')
k_nDGP_ma, Pk_nDGP_ma = load_lrg_pk_for_model('nDGP_N1.2', 's1234-fixed-inv')
assert np.allclose(k_nDGP_a, k_nDGP_ma), "nDGP_Abacus_a/_ma LRG k-grids differ"
k_nDGP_LRG = k_nDGP_a
Pk_nDGP_LRG_PairedFixed = (Pk_nDGP_a + Pk_nDGP_ma) / 2


# Plot 1

In [ ]:
# Four-panel LRG power spectrum comparison at z=0.500: LCDM (top row) vs nDGP (bottom row).
# Paired-fixed (mean of the "_a"/"_ma" phase pair) only -- no unfixed realisation, no 1-sigma
# band. Mirrors the layout of power_spectrum2tris_Abacus.ipynb's Plot 1, with LRG P(k) in
# place of matter P(k). There's no LRG N-body reference yet (Abacus1's halo catalog is still
# raw CompaSO/ASDF), so the COLA/Nbody ratio panel (top-right) is hidden, the same way
# power_spectrum2_Abacus.ipynb's Plot 6 hides its panels when there's no N-body comparison yet.
k_LCDM_LRG_plot = np.asarray(k_LCDM_LRG)
Pk_LCDM_LRG_plot = np.asarray(Pk_LCDM_LRG_PairedFixed)
k_nDGP_LRG_plot = np.asarray(k_nDGP_LRG)
Pk_nDGP_LRG_plot = np.asarray(Pk_nDGP_LRG_PairedFixed)

fig_size = (6.6, 4.35)
label_size = 10.5
tick_size = 8.8
legend_size = 8.3
line_width = 1.45
cola_color = '#c47f2b'
ref_color = '0.35'

fig, axes = plt.subplots(2, 2, figsize=fig_size, dpi=300, constrained_layout=True)
ax_tl, ax_tr = axes[0]
ax_bl, ax_br = axes[1]

# Top row: LCDM LRG P(k)
ax_tl.semilogx(k_LCDM_LRG_plot, k_LCDM_LRG_plot**(3/2) * Pk_LCDM_LRG_plot, lw=line_width, color=cola_color, ls='--', label=r'$\mathrm{COLA\ (paired-fixed)}$')
ax_tl.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_tl.set_ylabel(r'$k^{3/2} P_{\mathrm{LRG}}(k)$', fontsize=label_size)
ax_tl.set_title(r'$\Lambda\mathrm{CDM}$', fontsize=label_size)
ax_tl.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25)

# Top-right: no LRG N-body reference yet, so no COLA/Nbody ratio panel
ax_tr.set_visible(False)

# Bottom row: nDGP LRG P(k)
ax_bl.semilogx(k_nDGP_LRG_plot, k_nDGP_LRG_plot**(3/2) * Pk_nDGP_LRG_plot, lw=line_width, color=cola_color, ls='--', label=r'$\mathrm{COLA\ (paired-fixed)}$')
ax_bl.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_bl.set_ylabel(r'$k^{3/2} P_{\mathrm{LRG}}(k)$', fontsize=label_size)
ax_bl.set_title(r'$\mathrm{nDGP}$', fontsize=label_size)
ax_bl.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25)
ax_bl.set_xlim(ax_tl.get_xlim())

# Bottom-right: nDGP/LCDM LRG P(k) ratio, paired-fixed only
Pk_nDGP_LRG_on_LCDM = np.interp(k_LCDM_LRG_plot, k_nDGP_LRG_plot, Pk_nDGP_LRG_plot)
ax_br.semilogx(k_LCDM_LRG_plot, Pk_nDGP_LRG_on_LCDM / Pk_LCDM_LRG_plot, lw=line_width, color=cola_color, ls='--', label=r'$\mathrm{COLA\ (paired-fixed)}$')
ax_br.axhline(1.0, color=ref_color, lw=0.9, ls=':')
ax_br.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_br.set_ylabel(r'$P_{\mathrm{nDGP}} / P_{\mathrm{LCDM}}$', fontsize=label_size)
ax_br.set_title(r'$\mathrm{nDGP}/\Lambda\mathrm{CDM}$', fontsize=label_size)
ax_br.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25, loc='upper left')

for ax in (ax_tl, ax_bl, ax_br):
    ax.minorticks_on()
    ax.grid(True, which='major', ls=':', color='0.82', alpha=0.75, lw=0.45)
    ax.tick_params(axis='both', which='major', direction='in', top=True, right=True, labelsize=tick_size, length=3.4, width=0.7)
    ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True, labelsize=tick_size - 0.8, length=2.0, width=0.55)
    for spine in ax.spines.values():
        spine.set_linewidth(0.7)

plt.show()


# Plot 2: LRG bias ratio

In [ ]:
# Matter P(k) of the same paired-fixed runs and redshift as the LRG mocks above: Hi-COLA's
# measurement on each snapshot (pofk_z<z>.txt), averaged over the pair.
k_LCDM_DM, Pk_LCDM_DM_PairedFixed = an.load_hicola('Abacus_Small', 'LCDM', an.PAIRED_FIXED, 0.5, 'PS')
k_nDGP_DM, Pk_nDGP_DM_PairedFixed = an.load_hicola('Abacus_Small', 'nDGP_N1.2', an.PAIRED_FIXED, 0.5, 'PS')

In [ ]:
# Scale-dependent LRG bias, b^2(k) = P_LRG(k) / P_matter(k), paired-fixed only.
# The matter P(k) comes from a different power-spectrum estimator/mesh (COLA's internal
# measurer) than the LRG P(k) (nbodykit FFTPower, Nmesh=256), so mild estimator-driven
# systematics are possible near the LRG Nyquist frequency (~1.6 h/Mpc) -- informational,
# not a sign of a real physical feature there.
Pk_LCDM_DM_on_LRG = np.interp(k_LCDM_LRG_plot, k_LCDM_DM, Pk_LCDM_DM_PairedFixed)
Pk_nDGP_DM_on_LRG = np.interp(k_nDGP_LRG_plot, k_nDGP_DM, Pk_nDGP_DM_PairedFixed)

bias2_LCDM = Pk_LCDM_LRG_plot / Pk_LCDM_DM_on_LRG
bias2_nDGP = Pk_nDGP_LRG_plot / Pk_nDGP_DM_on_LRG

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=fig_size, dpi=300, constrained_layout=True)

ax_l.semilogx(k_LCDM_LRG_plot, bias2_LCDM, lw=line_width, color=cola_color, ls='--', label=r'$\mathrm{COLA\ (paired-fixed)}$')
ax_l.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_l.set_ylabel(r'$P_{\mathrm{LRG}}(k) / P_{\mathrm{m}}(k)$', fontsize=label_size)
ax_l.set_title(r'$\Lambda\mathrm{CDM}$', fontsize=label_size)
ax_l.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25)

ax_r.semilogx(k_nDGP_LRG_plot, bias2_nDGP, lw=line_width, color=cola_color, ls='--', label=r'$\mathrm{COLA\ (paired-fixed)}$')
ax_r.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_r.set_ylabel(r'$P_{\mathrm{LRG}}(k) / P_{\mathrm{m}}(k)$', fontsize=label_size)
ax_r.set_title(r'$\mathrm{nDGP}$', fontsize=label_size)
ax_r.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25)

for ax in (ax_l, ax_r):
    ax.minorticks_on()
    ax.grid(True, which='major', ls=':', color='0.82', alpha=0.75, lw=0.45)
    ax.tick_params(axis='both', which='major', direction='in', top=True, right=True, labelsize=tick_size, length=3.4, width=0.7)
    ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True, labelsize=tick_size - 0.8, length=2.0, width=0.55)
    for spine in ax.spines.values():
        spine.set_linewidth(0.7)

plt.show()

# Plot 3: LRG power spectrum, centrals only (no satellites)

In [ ]:
# Same paired-fixed boxes/mocks as Plot 1, but keeping only central galaxies -- drops the
# satellite population (and its extra small-scale power/shot noise) to isolate the
# central-galaxy-only clustering signal.
k_LCDM_a_cen, Pk_LCDM_a_cen = load_lrg_pk_for_model('LCDM', 's1234-fixed', centrals_only=True)
k_LCDM_ma_cen, Pk_LCDM_ma_cen = load_lrg_pk_for_model('LCDM', 's1234-fixed-inv', centrals_only=True)
assert np.allclose(k_LCDM_a_cen, k_LCDM_ma_cen), "LCDM_Abacus_a/_ma centrals-only k-grids differ"
k_LCDM_LRG_cen = k_LCDM_a_cen
Pk_LCDM_LRG_cen_PairedFixed = (Pk_LCDM_a_cen + Pk_LCDM_ma_cen) / 2

k_nDGP_a_cen, Pk_nDGP_a_cen = load_lrg_pk_for_model('nDGP_N1.2', 's1234-fixed', centrals_only=True)
k_nDGP_ma_cen, Pk_nDGP_ma_cen = load_lrg_pk_for_model('nDGP_N1.2', 's1234-fixed-inv', centrals_only=True)
assert np.allclose(k_nDGP_a_cen, k_nDGP_ma_cen), "nDGP_Abacus_a/_ma centrals-only k-grids differ"
k_nDGP_LRG_cen = k_nDGP_a_cen
Pk_nDGP_LRG_cen_PairedFixed = (Pk_nDGP_a_cen + Pk_nDGP_ma_cen) / 2

In [ ]:
# Same three-panel layout as Plot 1 (top-right hidden, no LRG N-body reference yet), but
# using centrals-only LRG P(k) instead of the full (centrals + satellites) sample.
k_LCDM_LRG_cen_plot = np.asarray(k_LCDM_LRG_cen)
Pk_LCDM_LRG_cen_plot = np.asarray(Pk_LCDM_LRG_cen_PairedFixed)
k_nDGP_LRG_cen_plot = np.asarray(k_nDGP_LRG_cen)
Pk_nDGP_LRG_cen_plot = np.asarray(Pk_nDGP_LRG_cen_PairedFixed)

fig, axes = plt.subplots(2, 2, figsize=fig_size, dpi=300, constrained_layout=True)
ax_tl, ax_tr = axes[0]
ax_bl, ax_br = axes[1]

# Top row: LCDM LRG P(k), centrals only
ax_tl.semilogx(k_LCDM_LRG_cen_plot, k_LCDM_LRG_cen_plot**(3/2) * Pk_LCDM_LRG_cen_plot, lw=line_width, color=cola_color, ls='--', label=r'$\mathrm{COLA\ (paired-fixed,\ centrals)}$')
ax_tl.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_tl.set_ylabel(r'$k^{3/2} P_{\mathrm{LRG,\,cen}}(k)$', fontsize=label_size)
ax_tl.set_title(r'$\Lambda\mathrm{CDM}$', fontsize=label_size)
ax_tl.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25)

# Top-right: no LRG N-body reference yet, so no COLA/Nbody ratio panel
ax_tr.set_visible(False)

# Bottom row: nDGP LRG P(k), centrals only
ax_bl.semilogx(k_nDGP_LRG_cen_plot, k_nDGP_LRG_cen_plot**(3/2) * Pk_nDGP_LRG_cen_plot, lw=line_width, color=cola_color, ls='--', label=r'$\mathrm{COLA\ (paired-fixed,\ centrals)}$')
ax_bl.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_bl.set_ylabel(r'$k^{3/2} P_{\mathrm{LRG,\,cen}}(k)$', fontsize=label_size)
ax_bl.set_title(r'$\mathrm{nDGP}$', fontsize=label_size)
ax_bl.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25)
ax_bl.set_xlim(ax_tl.get_xlim())

# Bottom-right: nDGP/LCDM LRG P(k) ratio, centrals only, paired-fixed
Pk_nDGP_LRG_cen_on_LCDM = np.interp(k_LCDM_LRG_cen_plot, k_nDGP_LRG_cen_plot, Pk_nDGP_LRG_cen_plot)
ax_br.semilogx(k_LCDM_LRG_cen_plot, Pk_nDGP_LRG_cen_on_LCDM / Pk_LCDM_LRG_cen_plot, lw=line_width, color=cola_color, ls='--', label=r'$\mathrm{COLA\ (paired-fixed,\ centrals)}$')
ax_br.axhline(1.0, color=ref_color, lw=0.9, ls=':')
ax_br.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_br.set_ylim(0.85, 1.15)
ax_br.set_ylabel(r'$P_{\mathrm{nDGP}} / P_{\mathrm{LCDM}}$', fontsize=label_size)
ax_br.set_title(r'$\mathrm{nDGP}/\Lambda\mathrm{CDM}$', fontsize=label_size)
ax_br.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25, loc='upper left')

for ax in (ax_tl, ax_bl, ax_br):
    ax.minorticks_on()
    ax.grid(True, which='major', ls=':', color='0.82', alpha=0.75, lw=0.45)
    ax.tick_params(axis='both', which='major', direction='in', top=True, right=True, labelsize=tick_size, length=3.4, width=0.7)
    ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True, labelsize=tick_size - 0.8, length=2.0, width=0.55)
    for spine in ax.spines.values():
        spine.set_linewidth(0.7)

plt.show()

# Plot 4: LRG redshift-space power spectrum multipoles (RSD)

In [ ]:
from HODDIES.utils import apply_rsd


def compute_lrg_rsd_multipoles(cat, HOD_obj, redshift, tracer='LRG', nmesh=256, dk=None, kmin=0.0,
                                 los='z', Nmu=5, subtract_shotnoise=True):
    # RSD displacement via HODDIES' own apply_rsd (the same routine HOD_obj.get_2PCF uses
    # internally): Delta_x = v_los * (1+z) / (H0 * E(z)) along a fixed box axis. Full LRG
    # sample (centrals + satellites), same as Plot 1 -- not the centrals-only Plot 3.
    mask = cat['TRACER'] == tracer
    boxsize = HOD_obj.boxsize
    pos_rsd = apply_rsd(cat[mask], redshift, boxsize, HOD_obj.cosmo, H_0=100, los=los, vsmear=0)
    pos = np.column_stack(pos_rsd).astype('f4')

    galaxy_cat = ArrayCatalog({'Position': pos})
    mesh = galaxy_cat.to_mesh(Nmesh=nmesh, BoxSize=boxsize, resampler='tsc',
                               interlaced=True, compensated=True)
    los_vec = [1 if ax == los else 0 for ax in 'xyz']
    result = FFTPower(mesh, mode='2d', dk=dk or 2 * np.pi / boxsize, kmin=kmin,
                       poles=[0, 2], los=los_vec, Nmu=Nmu)

    k = result.poles['k']
    P0 = result.poles['power_0'].real
    P2 = result.poles['power_2'].real
    if subtract_shotnoise:
        # Shot noise is isotropic -- it only contributes to the monopole (ell=0).
        P0 = P0 - result.poles.attrs['shotnoise']

    valid = k > 0  # drop the k=0 DC bin
    return k[valid], P0[valid], P2[valid]


def load_lrg_rsd_multipoles_for_model(model, realization, boxsize=500.0, redshift=0.500, fix_seed=42):
    # Same halo-catalog loading / cosmology setup as load_lrg_pk_for_model, but keeps
    # HOD_obj and cat around (instead of collapsing straight to a real-space P(k)) since
    # the RSD step needs the galaxy velocities and HOD_obj.cosmo.
    param_file = str(hp.NOTEBOOK_DIR / f'HOD_parameter_file_z{redshift:.3f}.yaml')
    output_dir = hp.halo_catalog_dir('Abacus_Small', model, realization, redshift)
    hoddies_catalog = output_dir / 'halos200c_hoddies.npy'
    halo_metadata_file = Path(str(output_dir / 'halos200c_catalog.npy') + '.json')

    if not hoddies_catalog.exists():
        raise FileNotFoundError(
            f'Missing {hoddies_catalog}. Run jobs/halo_finder.sh on '
            f'Abacus_Small/{model}_{realization}.lua first.'
        )

    arr = np.load(hoddies_catalog)

    if halo_metadata_file.exists():
        with halo_metadata_file.open() as f:
            halo_metadata = json.load(f)
        require_peculiar_velocities(halo_metadata, halo_metadata_file)
        halo_cosmology = halo_metadata.get('cosmology')
        if halo_cosmology is not None:
            mismatches = {
                key: (halo_cosmology.get(key), expected)
                for key, expected in SERIALIZABLE_PARAMETERS.items()
                if key in halo_cosmology and not np.isclose(float(halo_cosmology[key]), float(expected))
            }
            if mismatches:
                raise ValueError(f'{model}/{realization}: halo catalog cosmology does not match AbacusSummit c000 reference: {mismatches}')

    base_catalog = Base_catalogue({name: arr[name] for name in arr.dtype.names}, boxsize=boxsize, z_simu=redshift)
    HOD_obj = HOD(base_catalog, param_file=param_file)
    HOD_obj.cosmo = abacus_cosmo  # used for the RSD of get_wp / get_xiells

    cat = HOD_obj.make_mock_cat(tracers='LRG', fix_seed=fix_seed)
    k, P0, P2 = compute_lrg_rsd_multipoles(cat, HOD_obj, redshift, tracer='LRG')
    return k, P0, P2

In [ ]:
k_LCDM_a_rsd, P0_LCDM_a, P2_LCDM_a = load_lrg_rsd_multipoles_for_model('LCDM', 's1234-fixed')
k_LCDM_ma_rsd, P0_LCDM_ma, P2_LCDM_ma = load_lrg_rsd_multipoles_for_model('LCDM', 's1234-fixed-inv')
assert np.allclose(k_LCDM_a_rsd, k_LCDM_ma_rsd), "LCDM_Abacus_a/_ma RSD k-grids differ"
k_LCDM_rsd = k_LCDM_a_rsd
P0_LCDM_PairedFixed = (P0_LCDM_a + P0_LCDM_ma) / 2
P2_LCDM_PairedFixed = (P2_LCDM_a + P2_LCDM_ma) / 2

k_nDGP_a_rsd, P0_nDGP_a, P2_nDGP_a = load_lrg_rsd_multipoles_for_model('nDGP_N1.2', 's1234-fixed')
k_nDGP_ma_rsd, P0_nDGP_ma, P2_nDGP_ma = load_lrg_rsd_multipoles_for_model('nDGP_N1.2', 's1234-fixed-inv')
assert np.allclose(k_nDGP_a_rsd, k_nDGP_ma_rsd), "nDGP_Abacus_a/_ma RSD k-grids differ"
k_nDGP_rsd = k_nDGP_a_rsd
P0_nDGP_PairedFixed = (P0_nDGP_a + P0_nDGP_ma) / 2
P2_nDGP_PairedFixed = (P2_nDGP_a + P2_nDGP_ma) / 2

In [ ]:
# Same three-panel layout as Plot 1/3 (top-right hidden, no LRG N-body reference yet), but
# now in redshift space: each panel shows the monopole (ell=0) and quadrupole (ell=2)
# multipoles instead of a single real-space P(k). Full LRG sample (centrals + satellites),
# paired-fixed only. Plotted as k^{3/2} P_ell(k), same scaling as Plot 1/3, so the ratio
# panel is unaffected (the k^{3/2} factor cancels in the ratio).
k_LCDM_rsd_plot = np.asarray(k_LCDM_rsd)
P0_LCDM_rsd_plot = np.asarray(P0_LCDM_PairedFixed)
P2_LCDM_rsd_plot = np.asarray(P2_LCDM_PairedFixed)
k_nDGP_rsd_plot = np.asarray(k_nDGP_rsd)
P0_nDGP_rsd_plot = np.asarray(P0_nDGP_PairedFixed)
P2_nDGP_rsd_plot = np.asarray(P2_nDGP_PairedFixed)

p2_color = '#3d7a5c'

fig, axes = plt.subplots(2, 2, figsize=fig_size, dpi=300, constrained_layout=True)
ax_tl, ax_tr = axes[0]
ax_bl, ax_br = axes[1]

# Top row: LCDM multipoles
ax_tl.semilogx(k_LCDM_rsd_plot, k_LCDM_rsd_plot**(3/2) * P0_LCDM_rsd_plot, lw=line_width, color=cola_color, ls='--', label=r'$P_0\ \mathrm{(paired-fixed)}$')
ax_tl.semilogx(k_LCDM_rsd_plot, k_LCDM_rsd_plot**(3/2) * P2_LCDM_rsd_plot, lw=line_width, color=p2_color, ls='-.', label=r'$P_2\ \mathrm{(paired-fixed)}$')
ax_tl.axhline(0.0, color=ref_color, lw=0.8, ls=':')
ax_tl.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_tl.set_ylabel(r'$k^{3/2} P_{\ell,\,\mathrm{LRG}}(k)$', fontsize=label_size)
ax_tl.set_title(r'$\Lambda\mathrm{CDM}$', fontsize=label_size)
ax_tl.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25)

# Top-right: no LRG N-body reference yet, so no COLA/Nbody ratio panel
ax_tr.set_visible(False)

# Bottom row: nDGP multipoles
ax_bl.semilogx(k_nDGP_rsd_plot, k_nDGP_rsd_plot**(3/2) * P0_nDGP_rsd_plot, lw=line_width, color=cola_color, ls='--', label=r'$P_0\ \mathrm{(paired-fixed)}$')
ax_bl.semilogx(k_nDGP_rsd_plot, k_nDGP_rsd_plot**(3/2) * P2_nDGP_rsd_plot, lw=line_width, color=p2_color, ls='-.', label=r'$P_2\ \mathrm{(paired-fixed)}$')
ax_bl.axhline(0.0, color=ref_color, lw=0.8, ls=':')
ax_bl.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_bl.set_ylabel(r'$k^{3/2} P_{\ell,\,\mathrm{LRG}}(k)$', fontsize=label_size)
ax_bl.set_title(r'$\mathrm{nDGP}$', fontsize=label_size)
ax_bl.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25)
ax_bl.set_xlim(ax_tl.get_xlim())

# Bottom-right: nDGP/LCDM ratio, per multipole, paired-fixed only
P0_nDGP_on_LCDM = np.interp(k_LCDM_rsd_plot, k_nDGP_rsd_plot, P0_nDGP_rsd_plot)
P2_nDGP_on_LCDM = np.interp(k_LCDM_rsd_plot, k_nDGP_rsd_plot, P2_nDGP_rsd_plot)
ax_br.semilogx(k_LCDM_rsd_plot, P0_nDGP_on_LCDM / P0_LCDM_rsd_plot, lw=line_width, color=cola_color, ls='--', label=r'$P_0$')
ax_br.semilogx(k_LCDM_rsd_plot, P2_nDGP_on_LCDM / P2_LCDM_rsd_plot, lw=line_width, color=p2_color, ls='-.', label=r'$P_2$')
ax_br.axhline(1.0, color=ref_color, lw=0.9, ls=':')
ax_br.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_br.set_ylabel(r'$P_{\ell,\,\mathrm{nDGP}} / P_{\ell,\,\mathrm{LCDM}}$', fontsize=label_size)
ax_br.set_ylim(0.85, 1.15)
ax_br.set_title(r'$\mathrm{nDGP}/\Lambda\mathrm{CDM}$', fontsize=label_size)
ax_br.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25, loc='upper left')

for ax in (ax_tl, ax_bl, ax_br):
    ax.minorticks_on()
    ax.grid(True, which='major', ls=':', color='0.82', alpha=0.75, lw=0.45)
    ax.tick_params(axis='both', which='major', direction='in', top=True, right=True, labelsize=tick_size, length=3.4, width=0.7)
    ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True, labelsize=tick_size - 0.8, length=2.0, width=0.55)
    for spine in ax.spines.values():
        spine.set_linewidth(0.7)

plt.show()